In [1]:
import numpy as np
import pandas as pd

In [2]:
df_chicago = pd.read_csv('../data/raw/Crimes_-_2001_to_Present_20260508.csv')
df_nypd = pd.read_csv('../data/raw/NYPD_Arrests_Data_(Historic)_20260508.csv')
df_la1 = pd.read_csv('../data/raw/Arrest_Data_from_2010_to_2019_20260508.csv')
df_la2 = pd.read_csv('../data/raw/Arrest_Data_from_2020_to_4_30_2025_20260508.csv')
df_la = pd.concat([df_la1,df_la2],ignore_index=True)

print(df_chicago.columns)
print(df_nypd.columns)
print(df_la.columns)
df_chicago.head(1),df_nypd.head(1),df_la.head(1)

Index(['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type',
       'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat',
       'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate',
       'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude',
       'Location'],
      dtype='object')
Index(['ARREST_KEY', 'ARREST_DATE', 'PD_CD', 'PD_DESC', 'KY_CD', 'OFNS_DESC',
       'LAW_CODE', 'LAW_CAT_CD', 'ARREST_BORO', 'ARREST_PRECINCT',
       'JURISDICTION_CODE', 'AGE_GROUP', 'PERP_SEX', 'PERP_RACE', 'X_COORD_CD',
       'Y_COORD_CD', 'Latitude', 'Longitude', 'Lon_Lat'],
      dtype='object')
Index(['Report ID', 'Report Type', 'Arrest Date', 'Time', 'Area ID',
       'Area Name', 'Reporting District', 'Age', 'Sex Code', 'Descent Code',
       'Charge Group Code', 'Charge Group Description', 'Arrest Type Code',
       'Charge', 'Charge Description', 'Disposition Description', 'Address',
       'Cross Street', 'LAT', 'LON', 'Location', 'Booking Date',
 

(         ID Case Number                    Date              Block  IUCR  \
 0  14071594    JK100608  12/31/2025 11:45:00 PM  006XX E GRAND AVE  1150   
 
          Primary Type        Description Location Description  Arrest  \
 0  DECEPTIVE PRACTICE  CREDIT CARD FRAUD      OTHER (SPECIFY)   False   
 
    Domestic  ...  Ward  Community Area  FBI Code  X Coordinate Y Coordinate  \
 0     False  ...  42.0             8.0        11     1180796.0    1904058.0   
 
    Year              Updated On  Latitude  Longitude  \
 0  2025  03/14/2026 03:41:39 PM  41.89199 -87.611462   
 
                         Location  
 0  (41.891990384, -87.611461502)  
 
 [1 rows x 22 columns],
    ARREST_KEY ARREST_DATE  PD_CD    PD_DESC  KY_CD  \
 0   318178990  12/31/2025  101.0  ASSAULT 3  344.0   
 
                       OFNS_DESC    LAW_CODE LAW_CAT_CD ARREST_BORO  \
 0  ASSAULT 3 & RELATED OFFENSES  PL 1200001          M           B   
 
    ARREST_PRECINCT  JURISDICTION_CODE AGE_GROUP PERP_SEX PERP

In [4]:
# Chicago
df_chicago['Date'] = pd.to_datetime(df_chicago['Date'])
df_chicago_clean = df_chicago[[
    'Date','Primary Type','Description',
    'Location Description','Arrest','Latitude','Longitude'
]].rename(columns={
    'Date':'datetime',
    'Primary Type':'crime_type',
    'Description':'description',
    'Location Description': 'location_desc',
    'Arrest':'arrested',
    'Latitude':'latitude',
    'Longitude':'longitude'
})
df_chicago_clean['date'] = df_chicago_clean['datetime'].dt.date
df_chicago_clean['time'] = df_chicago_clean['datetime'].dt.time
df_chicago_clean.drop(columns='datetime',inplace=True)
df_chicago_clean['city'] = 'Chicago'

# NYPD
df_nypd['ARREST_DATE'] = pd.to_datetime(df_nypd['ARREST_DATE'])
df_nypd_clean = df_nypd[[
    'ARREST_DATE','OFNS_DESC','PD_DESC',
    'PERP_RACE','PERP_SEX','AGE_GROUP',
    'Latitude','Longitude'
]].rename(columns={
    'ARREST_DATE':'date',
    'OFNS_DESC':'crime_type',
    'PD_DESC':'description',
    'PERP_RACE':'race',
    'PERP_SEX':'sex',
    'AGE_GROUP':'age_group',
    'Latitude':'latitude',
    'Longitude':'longitude'
})
df_nypd_clean['date'] = df_nypd_clean['date'].dt.date
df_nypd_clean['time'] = None
df_nypd_clean['location_desc'] = None
df_nypd_clean['arrested'] = True
df_nypd_clean['city'] = 'New York'

# LA
df_la['Arrest Date'] = pd.to_datetime(df_la['Arrest Date'],format='mixed')
df_la_clean = df_la[[
    'Arrest Date','Time','Charge Group Description','Charge Description',
    'Area Name','Descent Code','Sex Code','Age','LAT','LON'
]].rename(columns={
    'Arrest Date':'date',
    'Time':'time',
    'Charge Group Description':'crime_type',
    'Charge Description':'description',
    'Area Name':'location_desc',
    'Descent Code':'race',
    'Sex Code':'sex',
    'Age':'age_group',
    'LAT':'latitude',
    'LON':'longitude'
})
df_la_clean['date'] = df_la_clean['date'].dt.date
df_la_clean['arrested'] = True
df_la_clean['city'] = 'Los Angeles'

# Combine
df = pd.concat([df_chicago_clean, df_nypd_clean, df_la_clean],ignore_index=True)

C:\Users\user\AppData\Local\Temp\ipykernel_10804\526745390.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_chicago['Date'] = pd.to_datetime(df_chicago['Date'])


In [82]:
df.isna().sum()

crime_type         49580
description        49380
location_desc    1723924
arrested               0
latitude           30064
longitude          30064
date                   0
time             1713446
city                   0
race             1952974
sex              1952974
age_group        1952974
dtype: int64

In [83]:
# These guys are useless in the analysis if null
df = df.dropna(subset=['latitude','longitude','crime_type'])

df['location_desc'] = df['location_desc'].fillna('Unknown')
df['description'] = df['description'].fillna('Unknown')

In [84]:
# No point in filling the rest of the nulls
df.isna().sum()

crime_type             0
description            0
location_desc          0
arrested               0
latitude               0
longitude              0
date                   0
time             1712211
city                   0
race             1922914
sex              1922914
age_group        1922914
dtype: int64

In [85]:
df['date'] = pd.to_datetime(df['date'])

# Filter out 2020 and 2021 because covid will mess with data
df = df[df['date'].dt.year.isin([2018, 2019, 2022, 2023, 2024, 2025])]

In [86]:
df = df[~((df['latitude'] == 0) | (df['longitude'] == 0))]

In [87]:
df['crime_type'].value_counts()

crime_type
THEFT                                351623
BATTERY                              272383
ASSAULT 3 & RELATED OFFENSES         208796
CRIMINAL DAMAGE                      165882
PETIT LARCENY                        144375
                                      ...  
NYS LAWS-UNCLASSIFIED VIOLATION           3
FELONY SEX CRIMES                         3
FORTUNE TELLING                           2
UNDER THE INFLUENCE OF DRUGS              2
DISRUPTION OF A RELIGIOUS SERVICE         2
Name: count, Length: 139, dtype: int64

In [88]:
crime_mapping = {
    # VIOLENT
    'BATTERY': 'VIOLENT',
    'ASSAULT': 'VIOLENT',
    'ASSAULT 3 & RELATED OFFENSES': 'VIOLENT',
    'ROBBERY': 'VIOLENT',
    'HOMICIDE': 'VIOLENT',
    'MURDER & NON-NEGL. MANSLAUGHTER': 'VIOLENT',
    'MURDER & NON-NEGL. MANSLAUGHTE': 'VIOLENT',
    'KIDNAPPING': 'VIOLENT',
    'FELONY ASSAULT': 'VIOLENT',
    'SEX CRIMES': 'VIOLENT',
    'RAPE': 'VIOLENT',
    'FELONY SEX CRIMES': 'VIOLENT',
    'HUMAN TRAFFICKING': 'VIOLENT',
    'Aggravated Assault': 'VIOLENT',
    'Other Assaults': 'VIOLENT',
    'Robbery': 'VIOLENT',
    'OFFENSES AGAINST THE PERSON': 'VIOLENT',
    'CRIMINAL SEXUAL ASSAULT': 'VIOLENT',
    'SEX OFFENSE': 'VIOLENT',
    'OFFENSE INVOLVING CHILDREN': 'VIOLENT',
    'STALKING': 'VIOLENT',
    'Against Family/Child': 'VIOLENT',
    'CRIM SEXUAL ASSAULT': 'VIOLENT',
    'Sex (except rape/prst)': 'VIOLENT',

    # PROPERTY
    'THEFT': 'PROPERTY',
    'BURGLARY': 'PROPERTY',
    'MOTOR VEHICLE THEFT': 'PROPERTY',
    'CRIMINAL DAMAGE': 'PROPERTY',
    'ARSON': 'PROPERTY',
    'PETIT LARCENY': 'PROPERTY',
    'GRAND LARCENY': 'PROPERTY',
    'POSSESSION OF STOLEN PROPERTY': 'PROPERTY',
    'CRIMINAL MISCHIEF & RELATED OF': 'PROPERTY',
    'CRIMINAL MISCHIEF & RELATED OFFENSES': 'PROPERTY',
    'BURGLAR\'S TOOLS': 'PROPERTY',
    'OTHER OFFENSES RELATED TO THEFT': 'PROPERTY',
    'OTHER OFFENSES RELATED TO THEF': 'PROPERTY',
    'Larceny': 'PROPERTY',
    'Vehicle Theft': 'PROPERTY',
    'CRIMINAL TRESPASS': 'PROPERTY',
    'GRAND LARCENY OF MOTOR VEHICLE': 'PROPERTY',
    'Burglary': 'PROPERTY',
    'UNAUTHORIZED USE OF A VEHICLE': 'PROPERTY',
    'Receive Stolen Property': 'PROPERTY',

    # DRUG
    'NARCOTICS': 'DRUG',
    'DRUG ABUSE': 'DRUG',
    'DANGEROUS DRUGS': 'DRUG',
    'CANNABIS POSSESSION': 'DRUG',
    'OTHER NARCOTIC VIOLATION': 'DRUG',
    'Narcotic Drug Laws': 'DRUG',

    # WEAPONS
    'WEAPONS VIOLATION': 'WEAPONS',
    'DANGEROUS WEAPONS': 'WEAPONS',
    'Weapon (carry/poss)': 'WEAPONS',

    # FRAUD
    'DECEPTIVE PRACTICE': 'FRAUD',
    'FRAUD': 'FRAUD',
    'FORGERY': 'FRAUD',
    'FRAUDS': 'FRAUD',
    'OFFENSES INVOLVING FRAUD': 'FRAUD',
    'IDENTITY THEFT': 'FRAUD',
    'FALSIFYING BUSINESS RECORDS': 'FRAUD',
    'Fraud/Embezzlement': 'FRAUD',

    # PUBLIC ORDER
    'PROSTITUTION': 'PUBLIC ORDER',
    'DISORDERLY CONDUCT': 'PUBLIC ORDER',
    'LIQUOR LAW VIOLATION': 'PUBLIC ORDER',
    'GAMBLING': 'PUBLIC ORDER',
    'LOITERING': 'PUBLIC ORDER',
    'FORTUNE TELLING': 'PUBLIC ORDER',
    'DISRUPTION OF A RELIGIOUS SERVICE': 'PUBLIC ORDER',
    'UNDER THE INFLUENCE OF DRUGS': 'PUBLIC ORDER',
    'VEHICLE AND TRAFFIC LAWS': 'PUBLIC ORDER',
    'OTHER TRAFFIC INFRACTION': 'PUBLIC ORDER',
    'INTOXICATED & IMPAIRED DRIVING': 'PUBLIC ORDER',
    'INTOXICATED/IMPAIRED DRIVING': 'PUBLIC ORDER',
    'Driving Under Influence': 'PUBLIC ORDER',
    'Prostitution/Allied': 'PUBLIC ORDER',
    'PROSTITUTION & RELATED OFFENSES': 'PUBLIC ORDER',
    'OFF. AGNST PUB ORD SENSBLTY &': 'PUBLIC ORDER',
    'OFF. AGNST PUB ORD SENSBLTY & RGHTS TO PRIV': 'PUBLIC ORDER',
    'OFFENSES AGAINST PUBLIC ADMINI': 'PUBLIC ORDER',
    'OFFENSES AGAINST PUBLIC ADMINISTRATION': 'PUBLIC ORDER',
    'Moving Traffic Violations': 'PUBLIC ORDER',
    'Liquor Laws': 'PUBLIC ORDER',
    'Drunkeness': 'PUBLIC ORDER',
    'Disorderly Conduct': 'PUBLIC ORDER',
    'PUBLIC PEACE VIOLATION': 'PUBLIC ORDER',
    'FOR OTHER AUTHORITIES': 'PUBLIC ORDER',
    'INTERFERENCE WITH PUBLIC OFFICER': 'PUBLIC ORDER',
    'Gambling': 'PUBLIC ORDER',
    'ADMINISTRATIVE CODE': 'PUBLIC ORDER',

    # OTHER
    'OTHER OFFENSE': 'OTHER',
    'MISCELLANEOUS PENAL LAW': 'OTHER',
    'Miscellaneous Other Violations': 'OTHER',
    'OTHER STATE LAWS': 'OTHER',
    'NYS LAWS-UNCLASSIFIED FELONY': 'OTHER',
    'OTHER STATE LAWS (NON PENAL LAW)': 'OTHER',
    'OTHER STATE LAWS (NON PENAL LA)': 'OTHER',
}

df['crime_category'] = df['crime_type'].map(crime_mapping).fillna('OTHER')
print(df['crime_category'].value_counts())

crime_category
PROPERTY        1170265
VIOLENT         1027191
PUBLIC ORDER     318625
OTHER            293716
DRUG             210330
FRAUD            155417
WEAPONS          118899
Name: count, dtype: int64


In [96]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek

# Time of day from time column (Chicago + LA only)
df['hour'] = pd.to_datetime(df['time'].astype(str), format='%H:%M:%S', errors='coerce').dt.hour

# Fix LA time
def parse_la_time(t):
    try:
        t = int(float(t))
        return t // 100
    except:
        return None

la_mask = df['city'] == 'Los Angeles'
df.loc[la_mask, 'hour'] = df.loc[la_mask, 'time'].apply(parse_la_time)

In [97]:
df.head()

,crime_type,description,location_desc,arrested,latitude,longitude,date,time,city,race,sex,age_group,crime_category,hour,year,month,day_of_week
0,DECEPTIVE PRACTICE,CREDIT CARD FRAUD,OTHER (SPECIFY),False,41.891990,-87.611462,2025-12-31,23:45:00,Chicago,NaN,NaN,NaN,FRAUD,23.0,2025,12,2
1,THEFT,OVER $500,OTHER (SPECIFY),False,41.891990,-87.611462,2025-12-31,23:45:00,Chicago,NaN,NaN,NaN,PROPERTY,23.0,2025,12,2
2,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,True,41.898388,-87.655790,2025-12-31,23:44:00,Chicago,NaN,NaN,NaN,VIOLENT,23.0,2025,12,2
3,WEAPONS VIOLATION,UNLAWFUL POSSESSION - HANDGUN,STREET,True,41.764554,-87.692250,2025-12-31,23:40:00,Chicago,NaN,NaN,NaN,WEAPONS,23.0,2025,12,2
4,ROBBERY,STRONG ARM - NO WEAPON,RESIDENCE - PORCH / HALLWAY,False,41.770945,-87.588017,2025-12-31,23:40:00,Chicago,NaN,NaN,NaN,VIOLENT,23.0,2025,12,2


In [101]:
print(df['city'].value_counts())
print(df.groupby('city')['crime_category'].value_counts())

city
Chicago        1512072
New York       1416393
Los Angeles     365978
Name: count, dtype: int64
city         crime_category
Chicago      PROPERTY          715490
             VIOLENT           488859
             OTHER              99988
             FRAUD              99048
             DRUG               50408
             WEAPONS            42330
             PUBLIC ORDER       15949
Los Angeles  VIOLENT           100066
             PUBLIC ORDER       89385
             OTHER              67008
             PROPERTY           46934
             DRUG               39009
             WEAPONS            20017
             FRAUD               3559
New York     VIOLENT           438266
             PROPERTY          407841
             PUBLIC ORDER      213291
             OTHER             126720
             DRUG              120913
             WEAPONS            56552
             FRAUD              52810
Name: count, dtype: int64


In [37]:
df = pd.read_csv('../data/processed/cleaned_crime_data.csv')
df.head()

C:\Users\user\AppData\Local\Temp\ipykernel_10804\2126414252.py:1: DtypeWarning: Columns (7,9,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/cleaned_crime_data.csv')


,crime_type,description,location_desc,arrested,latitude,longitude,date,time,city,race,sex,age_group,crime_category,hour,year,month,day_of_week
0,DECEPTIVE PRACTICE,CREDIT CARD FRAUD,OTHER (SPECIFY),False,41.891990,-87.611462,2025-12-31,23:45:00,Chicago,NaN,NaN,NaN,FRAUD,23.0,2025,12,2
1,THEFT,OVER $500,OTHER (SPECIFY),False,41.891990,-87.611462,2025-12-31,23:45:00,Chicago,NaN,NaN,NaN,PROPERTY,23.0,2025,12,2
2,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,True,41.898388,-87.655790,2025-12-31,23:44:00,Chicago,NaN,NaN,NaN,VIOLENT,23.0,2025,12,2
3,WEAPONS VIOLATION,UNLAWFUL POSSESSION - HANDGUN,STREET,True,41.764554,-87.692250,2025-12-31,23:40:00,Chicago,NaN,NaN,NaN,WEAPONS,23.0,2025,12,2
4,ROBBERY,STRONG ARM - NO WEAPON,RESIDENCE - PORCH / HALLWAY,False,41.770945,-87.588017,2025-12-31,23:40:00,Chicago,NaN,NaN,NaN,VIOLENT,23.0,2025,12,2


In [36]:
df[df['race'].isna()]

,crime_type,description,location_desc,arrested,latitude,longitude,date,time,city,race,sex,age_group,crime_category,hour,year,month,day_of_week
0,DECEPTIVE PRACTICE,CREDIT CARD FRAUD,OTHER (SPECIFY),False,41.891990,-87.611462,2025-12-31,23:45:00,Chicago,NaN,NaN,NaN,FRAUD,23.0,2025,12,2
1,THEFT,OVER $500,OTHER (SPECIFY),False,41.891990,-87.611462,2025-12-31,23:45:00,Chicago,NaN,NaN,NaN,PROPERTY,23.0,2025,12,2
2,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,True,41.898388,-87.655790,2025-12-31,23:44:00,Chicago,NaN,NaN,NaN,VIOLENT,23.0,2025,12,2
3,WEAPONS VIOLATION,UNLAWFUL POSSESSION - HANDGUN,STREET,True,41.764554,-87.692250,2025-12-31,23:40:00,Chicago,NaN,NaN,NaN,WEAPONS,23.0,2025,12,2
4,ROBBERY,STRONG ARM - NO WEAPON,RESIDENCE - PORCH / HALLWAY,False,41.770945,-87.588017,2025-12-31,23:40:00,Chicago,NaN,NaN,NaN,VIOLENT,23.0,2025,12,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1512067,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT $300 AND UNDER,RESIDENCE,False,41.702750,-87.620719,2018-01-01,00:00:00,Chicago,NaN,NaN,NaN,FRAUD,0.0,2018,1,0
1512068,DECEPTIVE PRACTICE,FRAUD OR CONFIDENCE GAME,RESIDENCE,False,41.799579,-87.761282,2018-01-01,00:00:00,Chicago,NaN,NaN,NaN,FRAUD,0.0,2018,1,0
1512069,SEX OFFENSE,AGG CRIMINAL SEXUAL ABUSE,RESIDENCE,False,41.653447,-87.608635,2018-01-01,00:00:00,Chicago,NaN,NaN,NaN,VIOLENT,0.0,2018,1,0
1512070,CRIM SEXUAL ASSAULT,PREDATORY,RESIDENCE,False,41.775900,-87.611922,2018-01-01,00:00:00,Chicago,NaN,NaN,NaN,VIOLENT,0.0,2018,1,0


In [39]:
race_map = {
    # LA codes
    'B': 'BLACK',
    'W': 'WHITE',
    'H': 'HISPANIC',
    'A': 'ASIAN / PACIFIC ISLANDER',
    'C': 'ASIAN / PACIFIC ISLANDER',
    'F': 'ASIAN / PACIFIC ISLANDER',
    'J': 'ASIAN / PACIFIC ISLANDER',
    'K': 'ASIAN / PACIFIC ISLANDER',
    'L': 'ASIAN / PACIFIC ISLANDER',
    'P': 'ASIAN / PACIFIC ISLANDER',
    'V': 'ASIAN / PACIFIC ISLANDER',
    'Z': 'ASIAN / PACIFIC ISLANDER',
    'I': 'AMERICAN INDIAN/ALASKAN NATIVE',
    'U': 'UNKNOWN',
    'X': 'UNKNOWN',
    'O': 'UNKNOWN',
    'G': 'UNKNOWN',
    'S': 'UNKNOWN',
    'D': 'UNKNOWN',

    # NYC labels (standardize Hispanic)
    'BLACK': 'BLACK',
    'WHITE': 'WHITE',
    'BLACK HISPANIC': 'HISPANIC',
    'WHITE HISPANIC': 'HISPANIC',
    'HISPANIC': 'HISPANIC',

    # keep existing standardized values
    'ASIAN / PACIFIC ISLANDER': 'ASIAN / PACIFIC ISLANDER',
    'AMERICAN INDIAN/ALASKAN NATIVE': 'AMERICAN INDIAN/ALASKAN NATIVE',
    'UNKNOWN': 'UNKNOWN'
}

df['race'] = df['race'].map(race_map)

In [42]:
df['race'].unique()

array([nan, 'BLACK', 'WHITE', 'HISPANIC', 'ASIAN / PACIFIC ISLANDER',
       'UNKNOWN', 'AMERICAN INDIAN/ALASKAN NATIVE'], dtype=object)

In [43]:
df.to_csv('../data/processed/cleaned_crime_data.csv',index=False)

In [45]:
sampled_df = df.sample(n=100000, random_state=67)

In [46]:
sampled_df.head()

,crime_type,description,location_desc,arrested,latitude,longitude,date,time,city,race,sex,age_group,crime_category,hour,year,month,day_of_week
1220274,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,False,41.703774,-87.564456,2019-02-14,00:30:00,Chicago,NaN,NaN,NaN,VIOLENT,0.0,2019,2,3
1669394,OTHER STATE LAWS,"NY STATE LAWS,UNCLASSIFIED VIO",Unknown,True,40.670658,-73.957974,2025-06-02,NaN,New York,BLACK,(null),(null),OTHER,NaN,2025,6,0
2828823,PETIT LARCENY,"LARCENY,PETIT FROM OPEN AREAS,UNCLASSIFIED",Unknown,True,40.802620,-73.951088,2018-05-15,NaN,New York,BLACK,M,25-44,PROPERTY,NaN,2018,5,1
2250835,CRIMINAL MISCHIEF & RELATED OF,"CRIMINAL MISCHIEF,UNCLASSIFIED 4",Unknown,True,40.855900,-73.897694,2023-02-14,NaN,New York,BLACK,M,45-64,PROPERTY,NaN,2023,2,1
3184672,Moving Traffic Violations,DRIVE W/LIC SUSPEND/REVOKE 4 DRUGS/ALCOHL,Olympic,True,34.047200,-118.309000,2022-05-12,1935.0,Los Angeles,HISPANIC,M,40,PUBLIC ORDER,19.0,2022,5,3


In [47]:
sampled_df.to_csv('../data/processed/sampled_cleaned_crime_data.csv',index=False)
